In [1]:
import os
import requests

def get_wikipedia_parsed_html(page_title):
    # Official MediaWiki API
    # FIX: Use the actual API gateway endpoint instead of the main homepage portal
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "format": "json",
        "page": page_title,
        "prop": "text",          # Fetch clean, structural HTML
        "redirects": 1
    }

    headers = {
        "User-Agent": "MyDataScraperApp/1.0 (your_email@example.com)"
    }

    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        data = response.json()
        if "parse" in data:
            html_body = data["parse"]["text"]["*"]

            # Fix live images by explicitly applying the HTTPS protocol
            html_body = html_body.replace('src="//upload.wikimedia.org', 'src="https://wikimedia.org')
            html_body = html_body.replace('srcset="//upload.wikimedia.org', 'srcset="https://wikimedia.org')

            # Clean up tracking layout images
            html_body = html_body.replace('src="/static/', 'src="https://wikipedia.org')

            # Fix layout article navigation links
            html_body = html_body.replace('href="/wiki/', 'href="https://wikipedia.org')
            html_body = html_body.replace('href="#', 'href="https://wikipedia.org' + page_title.replace(' ', '_') + '#')
            return html_body
    return None

def build_wikipedia_reader_page(title, body_content):
    if not body_content:
        return "<h1>Error: Article structure could not be retrieved.</h1>"

    # Embedded presentation stylesheet
    html_document = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title} - Reader Layout</title>

    <!-- Load MathJax so mathematical equations and complex symbols render perfectly -->
    <script src="https://polyfill.io"></script>
    <script id="MathJax-script" async src="https://jsdelivr.net"></script>

    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            background-color: #f8f9fa;
            color: #202122;
            line-height: 1.6;
            margin: 0;
            padding: 40px 20px;
        }}
        .wiki-article-container {{
            max-width: 960px;
            margin: 0 auto;
            background: #ffffff;
            padding: 40px;
            border: 1px solid #a2a9b1;
            border-radius: 4px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.05);
        }}

        /* Main Heading Typography */
        h1.firstHeading {{
            font-family: 'Linux Libertine', 'Georgia', 'Times', serif;
            font-size: 2.5rem;
            font-weight: normal;
            margin-top: 0;
            margin-bottom: 20px;
            border-bottom: 1px solid #a2a9b1;
            padding-bottom: 5px;
        }}

        /* Structural Elements */
        h2 {{
            font-family: 'Linux Libertine', 'Georgia', 'Times', serif;
            font-size: 1.7rem;
            font-weight: normal;
            border-bottom: 1px solid #a2a9b1;
            margin-top: 40px;
            padding-bottom: 5px;
        }}
        h3 {{
            font-size: 1.3rem;
            margin-top: 30px;
        }}
        p {{
            margin: 0.6em 0 1em 0;
            font-size: 1.05rem;
            text-align: justify;
        }}

        /* Links and References Style */
        a {{
            color: #36b;
            text-decoration: none;
        }}
        a:hover {{
            text-decoration: underline;
        }}
        .mw-reflink, .reference {{
            font-size: 80%;
            font-weight: normal;
            vertical-align: super;
            line-height: 1;
            padding-left: 2px;
        }}

        /* Numbered & Bulleted Lists */
        ul, ol {{
            margin: 0.5em 0 1em 1.8em;
            padding: 0;
        }}
        li {{
            margin-bottom: 0.4em;
            font-size: 1.05rem;
        }}

        /* Images and Thumbnails Alignment */
        img {{
            max-width: 100%;
            height: auto;
        }}
        .thumb {{
            margin-bottom: 1.5em;
            border: 1px solid #c8ccd1;
            background-color: #f8f9fa;
            padding: 3px;
        }}
        .tright {{
            float: right;
            clear: right;
            margin-left: 1.4em;
        }}
        .tleft {{
            float: left;
            clear: left;
            margin-right: 1.4em;
        }}
        .thumbcaption {{
            border: none;
            font-size: 85%;
            line-height: 1.4;
            padding: 3px;
            color: #54595d;
        }}

        /* Infoboxes and Technical Tables */
        .infobox, .wikitable {{
            background-color: #f8f9fa;
            border: 1px solid #a2a9b1;
            border-collapse: collapse;
            margin: 1em 0 1em 1.4em;
            padding: 5px;
            float: right;
            clear: right;
            font-size: 88%;
            width: 300px;
        }}
        .wikitable th, .wikitable td {{
            border: 1px solid #a2a9b1;
            padding: 0.4em 0.6em;
        }}
        .wikitable th {{
            background-color: #eaecf0;
            text-align: center;
        }}

        /* Code snippet block styling */
        code, pre {{
            background-color: #f8f9fa;
            color: #000;
            border: 1px solid #eaecf0;
            border-radius: 3px;
            padding: 2px 5px;
            font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace;
            font-size: 90%;
        }}
        pre {{
            padding: 15px;
            overflow-x: auto;
            line-height: 1.5;
        }}

        /* Cleanup non-essential elements */
        .mw-editsection, .noprint, .mw-empty-elt, .mw-jump-link {{
            display: none !important;
        }}
    </style>
</head>
<body>
    <div class="wiki-article-container">
        <h1 class="firstHeading">{title}</h1>
        {body_content}
    </div>
</body>
</html>"""

    return html_document

if __name__ == "__main__":
    # Choose your targeted Wikipedia article name
    article_name = "Python (programming language)"
    print(f"Downloading completely styled archive for '{article_name}'...")

    # 1. Fetch raw, complete layout
    parsed_html = get_wikipedia_parsed_html(article_name)

    # 2. Re-map image sources and inject fonts/styles
    styled_document = build_wikipedia_reader_page(article_name, parsed_html)

    # 3. Create index file
    file_path = "index.html"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(styled_document)

    print(f"\nSuccess! Open '{os.path.abspath(file_path)}' directly in any browser to see text, lists, and images working together.")


Success! Open '/content/index.html' directly in any browser to see text, lists, and images working together.
